# New baselines, tier B -- discrete set-cover

Implements `TODO.md` item 5's second bullet: **discrete (hard) greedy set-cover**, the same
lazy-greedy *mechanism* as `tokenizer.LazyGreedyTokenSelector`, but with `S_h(u,T)` replaced
by a **binary** "is `u` within `D` hops of *some* token" indicator instead of the
averaged/soft Eq. 1 score -- structurally almost identical to the paper's own selector, so
it doubles as a literature baseline *and* an ablation of what the soft coverage formulation
buys over classic hard set-cover greedy (Nemhauser, Wolsey & Fisher 1978's `(1-1/e)`
guarantee, same as the paper's own method).

Item 5's *first* bullet, bounded k-center (Gonzalez 1985 farthest-first traversal), is **not**
implemented -- see `classical_selectors.py`'s module docstring and `TODO.md` item 5's note for
why: Gonzalez's 2-approximation guarantee needs a genuine metric (symmetric + triangle
inequality), but this graph is directed, and symmetrizing directed shortest-path distance via
`min(dist(u,v), dist(v,u))` does not preserve the triangle inequality in general -- and on this
project's actual graph the D-hop relation behaves like a DAG in practice (of 1,468,867
within-D pairs, zero have both directions reachable), so the symmetrization would have been
silently degenerating to "whichever single direction happens to exist," not a real distance.

**Implementation note**: `HardGreedySetCoverSelector` lives in a new module,
`src/graph_tokenizer_gd_tree_dev/classical_selectors.py`, *not* `tokenizer.py` -- it's
self-contained (operates directly on the precomputed `all_rel_distance_subgraph.parquet` D-hop
distance table from notebook 3, not tokenizer.py's matrix-based Eq. 1 machinery) and doesn't
need or touch any of `tokenizer.py`'s internals. Because it's *adaptive* (each pick depends on
what's already been picked, unlike the one-shot rankings in
`3.baseline_candidate_selection.ipynb`/`5.3.centrality_baselines.ipynb`), it returns an ordered
`(token, gain, cumulative_score)` history -- the same schema
`2.greedy_tree_candidate_seledction.ipynb` already writes for `greedy_tree_margin` -- saved here
as a new `file_type` under `baseline_candidates/`, so `4.comparison_candidate_set.ipynb`'s
glob-based discovery (and `drilldown.py`/`app_new_tokenizer.py`) picks it up automatically, no
code changes needed there.

The selector doesn't need `combined_subgraphs.gpickle` itself -- only the precomputed distance
table and `mapped_ids` -- so this notebook skips loading the ~75k-node graph pickle entirely.

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import pickle

import polars as pl

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.classical_selectors as classical_selectors

## Load mapped concepts, `id_to_label`, and the D-hop distance table

In [3]:
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(config.BasicConfig().mapped_path)
mapped_ids = sorted(df_mapped["id"].unique().to_list())
D = config.TokenizerParam().max_dist_candidate

# all (src_id, dst_id, distance) pairs in combined_subgraphs -- built once in
# 3.baseline_candidate_selection.ipynb, already used by highest_degree/most_children.
distance_df = pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)

# Same "rank as many candidates as the greedy selector does" convention notebook 2 uses
# (k = len(mapped_ids)) -- comfortably above max(Ks) = 11500, the largest k the eval sweep
# ever needs, regardless of how many candidates the selector actually manages to rank.
K_TARGET = len(mapped_ids)
print(f"mapped_ids: {len(mapped_ids):,}   D: {D}   K_TARGET: {K_TARGET:,}   max(Ks): {config.TokenizerParam().Ks.max()}")

mapped_ids: 46,150   D: 3   K_TARGET: 46,150   max(Ks): 11500


In [4]:
distance_df

src_id,dst_id,distance
str,str,i64
"""472889002""","""285570007""",1
"""472889002""","""257261003""",1
"""472889002""","""12921003""",1
"""472889002""","""430232008""",1
"""472889002""","""257915005""",2
…,…,…
"""417746004+724930001+724932009""","""417746004+724930001+724932009""",0
"""22525001""","""22525001""",0
"""724932009""","""724932009""",0


## Discrete (hard) greedy set-cover

`HardGreedySetCoverSelector` covers concept `c` with token `t` iff `t` appears as a
`(src_id=c, dst_id=t, distance<=D)` row in the table above -- the same D-hop relation
`highest_degree` already ranks by, just restricted to `src_id` in `M` and made binary instead
of averaged. `select()` runs the classic lazy/CELF max-coverage greedy loop to completion (or
`K_TARGET`, whichever is smaller) and returns the pick order.

In [4]:
set_cover_selector = classical_selectors.HardGreedySetCoverSelector(distance_df, mapped_ids, D)
history_set_cover = set_cover_selector.select(k=K_TARGET, verbose=True, progress_every=5000)

print(f"\nranked {len(history_set_cover):,} candidates "
      f"(of {len(set_cover_selector.covers):,} that cover >=1 mapped concept; "
      f"{len(set_cover_selector._leftover):,} zero-coverage candidates padded at the tail)")

df_set_cover = (
    pl.DataFrame(history_set_cover, schema=["token", "gain", "cumulative_score"], orient="row")
    .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
    .with_row_index()
)
df_set_cover.write_parquet(config.CandidateLists().discrete_set_cover)
df_set_cover.head()

[  5000/46150] token='126068008'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 10000/46150] token='226351000'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 15000/46150] token='281246002'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 20000/46150] token='36462009'      gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 25000/46150] token='419397004'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 30000/46150] token='61747003'      gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 35000/46150] token='774587000'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)

ranked 38,738 candidates (of 38,738 that cover >=1 mapped concept; 0 zero-coverage candidates padded at the tail)


index,token,gain,cumulative_score,label
u32,str,i64,f64,str
0,"""129264002""",20472,0.443597,"""Action (qualifier value)"""
1,"""182353008""",5240,0.55714,"""Side (qualifier value)"""
2,"""64572001""",2982,0.621755,"""Disease (disorder)"""
3,"""129265001""",2028,0.665699,"""Evaluation - action (qualifier…"
4,"""49755003""",1658,0.701625,"""Morphologically abnormal struc…"


## Next steps

One new file now exists under `baseline_candidates/`: `discrete_set_cover.parquet`. Re-run
`4.comparison_candidate_set.ipynb` to score it across the usual `k` sweep and get it into the
comparison plots/tables and `app_new_tokenizer.py` alongside the existing baselines -- no code
changes needed there, since both use glob-based discovery over everything in
`baseline_candidates/`.

Reading `gain`/`cumulative_score`: they are **not** on the same scale as
`greedy_tree_margin`'s `gain`/`cumulative_score` -- see `classical_selectors.py`'s class
docstring. `cumulative_score` here is a hard 0/1 coverage fraction in `(0, 1]`, comparable in
*scale* (not value) to `semantic_coverage` -- don't average or compare it directly against
other file_types' columns; use `eval.py`'s metrics (computed uniformly for every candidate
list in notebook 4) for that.